In [1]:
import os
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense

In [2]:
# Cargar dataset MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalizar imágenes
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [3]:
# Crear modelo
model=tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10,activation="softmax")
])

c:\Users\otice\Documents\9no semestre\Inteligencia Artificial 2\venvirtual\flow\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
#Compilar modelo
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [5]:
# Mostrar arquitectura
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
#Entrenamiento
history = model.fit(
    x_train,
    y_train,
    epochs=6,
    batch_size=128,
    validation_split=0.1,
    verbose=2
)

Epoch 1/6
422/422 - 9s - 20ms/step - accuracy: 0.8995 - loss: 0.3466 - val_accuracy: 0.9572 - val_loss: 0.1493
Epoch 2/6
422/422 - 7s - 16ms/step - accuracy: 0.9593 - loss: 0.1376 - val_accuracy: 0.9703 - val_loss: 0.1082
Epoch 3/6
422/422 - 5s - 12ms/step - accuracy: 0.9715 - loss: 0.0962 - val_accuracy: 0.9717 - val_loss: 0.0969
Epoch 4/6
422/422 - 7s - 16ms/step - accuracy: 0.9782 - loss: 0.0722 - val_accuracy: 0.9760 - val_loss: 0.0863
Epoch 5/6
422/422 - 4s - 9ms/step - accuracy: 0.9827 - loss: 0.0561 - val_accuracy: 0.9778 - val_loss: 0.0768
Epoch 6/6
422/422 - 6s - 15ms/step - accuracy: 0.9862 - loss: 0.0455 - val_accuracy: 0.9743 - val_loss: 0.0839


In [7]:
# Evaluación
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"\nLoss: {loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")


Loss: 0.0854
Accuracy: 0.9730


In [8]:
model.save("mnist_model.keras")

print("\nModelo guardado correctamente.")


Modelo guardado correctamente.


In [2]:
import threading
from flask import Flask, render_template, request, jsonify
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps
import io
import base64
import re

In [ ]:
# ======================================
# Crear aplicación Flask
# ======================================
app = Flask(__name__)

# ======================================
# Cargar modelo entrenado
# ======================================
MODEL_PATH = "mnist_model.keras"
model = tf.keras.models.load_model(MODEL_PATH)

# ======================================
# Preprocesamiento de la imagen
# ======================================
def preprocess_image_from_base64(b64data):

    match = re.search(r"base64,(.*)", b64data)
    if not match:
        raise ValueError("Imagen no válida")

    img_bytes = base64.b64decode(match.group(1))
    img = Image.open(io.BytesIO(img_bytes)).convert("L")

    # Invertir colores (negro sobre blanco -> blanco sobre negro)
    img = ImageOps.invert(img)

    # Convertir a arreglo para recortar el área dibujada
    arr = np.array(img)

    coords = np.column_stack(np.where(arr > 0))

    if coords.size > 0:

        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)

        arr = arr[y_min:y_max+1, x_min:x_max+1]

    else:

        arr = np.zeros((28,28), dtype=np.uint8)

    # Redimensionar manteniendo la relación de aspecto
    img = Image.fromarray(arr)
    img.thumbnail((20,20), Image.Resampling.LANCZOS)

    # Crear un lienzo de 28x28
    canvas = Image.new("L",(28,28),color=0)

    paste_x = (28-img.width)//2
    paste_y = (28-img.height)//2

    canvas.paste(img,(paste_x,paste_y))

    # Normalizar
    arr = np.array(canvas).astype("float32")/255.0

    # Forma esperada por el modelo: (1,28,28)
    arr = np.expand_dims(arr, axis=0)

    return arr


# ======================================
# Página principal
# ======================================
@app.route("/")
def index():
    return render_template("index.html")


# ======================================
# Predicción
# ======================================
@app.route("/predict", methods=["POST"])
def predict():

    try:

        data = request.get_json()

        img_data = data["image"]

        x = preprocess_image_from_base64(img_data)

        probabilities = model.predict(x, verbose=0)[0]

        prediction = int(np.argmax(probabilities))

        return jsonify({

            "prediction": prediction,

            "probabilities": [float(p) for p in probabilities]

        })

    except Exception as e:

        return jsonify({"error": str(e)}),400


# ======================================
# Ejecutar servidor Flask
# ======================================
def run_flask():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


# ======================================
# Iniciar servidor en segundo plano
# ======================================
flask_thread = threading.Thread(target=run_flask)
flask_thread.start()

print("Servidor Flask iniciado correctamente.")
print("Abre tu navegador en:")
print("http://localhost:5000")

Servidor Flask iniciado correctamente.
Abre tu navegador en:
http://localhost:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.16.8.15:5000
Press CTRL+C to quit
